# CNN on Fashion-MNIST (GPU)

This notebook mirrors `gpu_notebook.ipynb` but trains a CNN on the Fashion-MNIST dataset using GPU (if available).

In [ ]:
# ============ Reproducibility ============
RANDOM_SEED = 42   # Change this value to get different random runs; fix it for reproducible results

import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print(f"Random seed set to {RANDOM_SEED}")

In [ ]:
import time
import torch
import torchvision
import torchvision.transforms as transforms

def data_loading(BATCH_SIZE, DOWNLOAD, SUBSET, seed=None):
    # Fashion-MNIST is grayscale (1 channel), 28x28
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=(-10, 10), translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_dataset = torchvision.datasets.FashionMNIST(
        root="./data_fashion",
        train=True,
        download=DOWNLOAD,
        transform=train_transform,
    )
    test_dataset = torchvision.datasets.FashionMNIST(
        root="./data_fashion",
        train=False,
        download=DOWNLOAD,
        transform=test_transform,
    )

    if SUBSET != 0:
        subset_indices = list(range(SUBSET))
        train_set = torch.utils.data.Subset(train_dataset, subset_indices)
        test_set = torch.utils.data.Subset(test_dataset, subset_indices)
        print(f"Using subset of {SUBSET} samples")
    else:
        train_set, test_set = train_dataset, test_dataset
        print("Using full Fashion-MNIST dataset")

    train_gen = torch.Generator().manual_seed(seed) if seed is not None else None
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=BATCH_SIZE, shuffle=True, generator=train_gen
    )
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, test_loader


def _evaluate(model, criterion, test_loader):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _test_acc, _test_err, _test_loss, total_test = 0, 0, 0.0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            _test_acc += (predicted == labels).sum().item()
            _test_err += (predicted != labels).sum().item()
            _test_loss += criterion(outputs, labels).item()

    test_loss = _test_loss / total_test
    test_err = 100 * _test_err / total_test
    test_acc = 100 * _test_acc / total_test
    return test_loss, test_err, test_acc


def train_and_evaluate(model, criterion, optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=90.0):
    train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values = [], [], [], [], [], [], [], []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    for i, epoch in enumerate(range(EPOCHS)):
        if i % 10 == 0:
            print(f"Epoch: {i+1}/{EPOCHS}")
        model.train()
        total_train, _train_err, _train_acc, running_loss = 0, 0, 0, 0.0
        _start = time.time()
        epoch_T_values = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            def closure(grad=True):
                output = model(images_s)
                loss = criterion(output, labels_s)
                if grad:
                    optimizer.zero_grad()
                    loss.backward()
                return loss

            if isinstance(optimizer, (torch.optim.SGD, torch.optim.Adam)):
                images_s = images
                labels_s = labels
                loss = closure()
                optimizer.step()
            else:
                if optimizer.reset is True:
                    images_s = images
                    labels_s = labels
                loss, T_value = optimizer.step(closure)
                T_val = float(T_value)
                T_values.append(T_val)
                epoch_T_values.append(T_val)

            running_loss += loss.item()
            output = model(images)
            _, predicted = torch.max(output.data, 1)
            total_train += labels.size(0)
            _train_err += (predicted != labels).sum().item()
            _train_acc += (predicted == labels).sum().item()

        run_time = time.time() - _start
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = 100 * _train_acc / total_train
        epoch_train_err = 100 * _train_err / total_train

        test_loss, test_err, test_acc = _evaluate(model, criterion, test_loader)

        train_losses.append(epoch_train_loss)
        train_errs.append(epoch_train_err)
        train_accs.append(epoch_train_acc)
        test_losses.append(test_loss)
        test_errs.append(test_err)
        test_accs.append(test_acc)
        run_times.append(run_time)

        if epoch % 10 == 0:
            log_msg = (
                f"E [{epoch+1}/{EPOCHS}]. train_loss_acc: {running_loss / len(train_loader):.4f}, {epoch_train_acc:.2f}%, "
                f"test_acc: {test_acc:.2f}%, run_time: {run_time}"
            )
            if epoch_T_values:
                mean_T = sum(epoch_T_values) / len(epoch_T_values)
                log_msg += f", T_mean: {mean_T:.2f}"
            print(log_msg)
        if early_stop and epoch_train_acc >= threshold:
            print(f"Early stopping at epoch {epoch+1} with train error {epoch_train_err:.2f}%")
            break

    return train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values

In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt

class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        # Input: 1x28x28
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)  # 32x28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 64x14x14 after pool
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)  # 64x7x7 after pool
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        # 1x28x28 -> 32x14x14
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        # 32x14x14 -> 64x7x7
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        # 64x7x7 -> 64x7x7
        x = self.conv3(x)
        x = self.relu(x)

        x = x.view(-1, 64 * 7 * 7)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


def metrics_plot(num_epochs, train_losses, test_losses, train_accs, test_accs, train_errs, test_errs):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 3, 1)
    plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss")
    plt.plot(range(1, num_epochs + 1), test_losses, label="Test Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Test Loss")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 2)
    plt.plot(range(1, num_epochs + 1), train_accs, label="Train Accuracy")
    plt.plot(range(1, num_epochs + 1), test_accs, label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("Training and Test Accuracy")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 3)
    plt.plot(range(1, num_epochs + 1), train_errs, label="Train Error")
    plt.plot(range(1, num_epochs + 1), test_errs, label="Test Error")
    plt.xlabel("Epoch")
    plt.ylabel("Error (%)")
    plt.title("Training and Test Error")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

BATCH_SIZE = 512
DOWNLOAD = True
SUBSET = 0  # set >0 for quick experiments

train_loader, test_loader = data_loading(BATCH_SIZE, DOWNLOAD, SUBSET, seed=RANDOM_SEED)

In [ ]:
from olnm import OLNM

set_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

criterion = nn.CrossEntropyLoss()

EPOCHS = 50
ADAM_LR = 0.001
SGD_LR = 0.1
OLNM_LR = 0.03
T = 100
results = {}

adam_model = FashionCNN().to(DEVICE)
sgd_model = FashionCNN().to(DEVICE)
olnm_model = FashionCNN().to(DEVICE)

adam_optimizer = optim.Adam(adam_model.parameters(), lr=ADAM_LR)
sgd_optimizer = optim.SGD(sgd_model.parameters(), lr=SGD_LR)
olnm_optimizer = OLNM(olnm_model.parameters(), lr=OLNM_LR, T=T, batch_size=BATCH_SIZE)

In [ ]:
# Train SGD on Fashion-MNIST
train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, _ = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, None)
train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, _ = train_and_evaluate(
    sgd_model, criterion, sgd_optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=95,
)
results["SGD"] = (
    train_losses, test_losses,
    train_errs, test_errs,
    train_accs, test_accs,
    run_times,
)
torch.cuda.empty_cache()

import json
with open("fashion_results_sgd.json", "w") as f:
    json.dump(results, f)

In [ ]:
# Train Adam on Fashion-MNIST
train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, _ = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, None)
train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, _ = train_and_evaluate(
    adam_model, criterion, adam_optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=95,
)
results["Adam"] = (
    train_losses, test_losses,
    train_errs, test_errs,
    train_accs, test_accs,
    run_times,
)
torch.cuda.empty_cache()

import json
with open("fashion_results_adam.json", "w") as f:
    json.dump(results, f)

In [ ]:
# Train OLNM on Fashion-MNIST
train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, [])
train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values = train_and_evaluate(
    olnm_model, criterion, olnm_optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=95,
)
results["OLNM"] = (
    train_losses, test_losses,
    train_errs, test_errs,
    train_accs, test_accs,
    run_times,
    T_values,
)
torch.cuda.empty_cache()

import json
with open("fashion_results_olnm.json", "w") as f:
    json.dump(results, f)